# Multiclass Evaluation & Metric Selection Lab

Extending binary classification metrics to multiple categories ($K > 2$) requires choosing an aggregation strategy: **Macro-averaging** (equal weight to each class), **Weighted-averaging** (weight proportional to class support), or **Micro-averaging** (equivalent to global sample-wise accuracy). This lab extracts One-vs-Rest metrics from $N \times N$ confusion matrices, benchmarks averaging schemes on balanced and imbalanced cohorts, and evaluates probabilistic scoring with Log Loss.

In [ ]:
import numpy as np
from sklearn.metrics import (
    confusion_matrix, precision_score, recall_score, f1_score,
    accuracy_score, classification_report, log_loss
)
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression

np.random.seed(42)
np.set_printoptions(precision=4, suppress=True)

## 1. Extracting One-vs-Rest Counts from an $N \times N$ Matrix

Fit a model on the 3-class Iris dataset and extract True Positives, False Positives, and False Negatives for each individual category.

In [ ]:
iris = load_iris()
X, y = iris.data, iris.target
clf = LogisticRegression(max_iter=1000, random_state=42).fit(X, y)
y_pred = clf.predict(X)

cm = confusion_matrix(y, y_pred)
print("Confusion Matrix (Rows=Actual, Columns=Predicted):")
print(cm)

print(f"\n{'Class Name':<16} {'TP':<5} {'FP':<5} {'FN':<5} {'Precision':<12} {'Recall':<12} {'F1-Score'}")
print("-" * 68)
for i, name in enumerate(iris.target_names):
    tp = cm[i, i]
    fp = cm[:, i].sum() - tp
    fn = cm[i, :].sum() - tp
    p = tp / (tp + fp) if (tp + fp) > 0 else 0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (p * r) / (p + r) if (p + r) > 0 else 0
    print(f"{name:<16} {tp:<5} {fp:<5} {fn:<5} {p:<12.1%} {r:<12.1%} {f1:.3f}")

## 2. Macro vs. Micro vs. Weighted Averaging on Imbalanced Data

Simulate a 3-class imbalanced cohort (Class 0: 60%, Class 1: 25%, Class 2: 15%) to compare how averaging methods respond to minority performance.

In [ ]:
y_imb = np.concatenate([np.zeros(60, dtype=int), np.ones(25, dtype=int), np.full(15, 2, dtype=int)])
y_pred_imb = y_imb.copy()
# Introduce errors focused on minority Class 2
y_pred_imb[85:] = 0  # Class 2 completely misclassified as Class 0

acc_val = accuracy_score(y_imb, y_pred_imb)
macro_f1 = f1_score(y_imb, y_pred_imb, average='macro')
weighted_f1 = f1_score(y_imb, y_pred_imb, average='weighted')
micro_f1 = f1_score(y_imb, y_pred_imb, average='micro')

print(f"Overall Accuracy: {acc_val:.1%}")
print(f"Micro F1:        {micro_f1:.3f} (Identical to accuracy)")
print(f"Weighted F1:     {weighted_f1:.3f} (Buoyed by 60% majority class)")
print(f"Macro F1:        {macro_f1:.3f} (Correctly penalizes 0% minority recall!)")
print("\nTakeaway: When rare classes fail, Macro F1 exposes the collapse while Weighted F1 hides it.")

## 3. Scikit-Learn Classification Report & Probabilistic Log Loss

Generate full diagnostic reports with `classification_report` and compute multiclass categorical cross-entropy via `log_loss`.

In [ ]:
print("Classification Report:")
print(classification_report(y, y_pred, target_names=iris.target_names))

probs = clf.predict_proba(X)
loss = log_loss(y, probs)
print(f"Multiclass Log Loss: {loss:.4f} (Evaluates calibrated confidence across all classes)")